In [ ]:
import os

os.makedirs("music", exist_ok=True)

with open("music/song1.txt", "w") as f:
    f.write("Sample Song 1 - Music File")

with open("music/song2.txt", "w") as f:
    f.write("Sample Song 2 - Music File")

with open("music/readme.txt", "w") as f:
    f.write("Welcome to the mini Spotify filesystem server.")

print("Music directory and sample files created.")

In [ ]:
import socket
import os

HOST = "127.0.0.1"
PORT = 5001
BASE_DIR = os.path.abspath("music")


def safe_path(filename):
    path = os.path.abspath(os.path.join(BASE_DIR, filename))

    if not path.startswith(BASE_DIR + os.sep):
        raise ValueError("Invalid file path")

    return path


def handle_command(command):
    parts = command.strip().split(maxsplit=1)

    if not parts:
        return "ERROR: Empty command"

    action = parts[0].upper()

    if action == "LIST":
        files = os.listdir(BASE_DIR)
        return "\n".join(files) if files else "Directory is empty"

    if action == "GET":
        if len(parts) != 2:
            return "ERROR: Filename required"

        try:
            path = safe_path(parts[1])

            if not os.path.isfile(path):
                return "ERROR: File not found"

            with open(path, "rb") as f:
                data = f.read()

            return data.decode("utf-8")

        except FileNotFoundError:
            return "ERROR: File not found"
        except Exception as e:
            return f"ERROR: {e}"

    if action == "DELETE":
        if len(parts) != 2:
            return "ERROR: Filename required"

        try:
            path = safe_path(parts[1])

            if not os.path.isfile(path):
                return "ERROR: File not found"

            os.remove(path)
            return f"SUCCESS: {parts[1]} deleted"

        except FileNotFoundError:
            return "ERROR: File not found"
        except Exception as e:
            return f"ERROR: {e}"

    return "ERROR: Unknown command"


def start_server():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as server:
        server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        server.bind((HOST, PORT))
        server.listen()

        print(f"Filesystem MCP Server running on {HOST}:{PORT}")

        while True:
            conn, address = server.accept()

            print(f"Client connected: {address}")

            with conn:
                command = conn.recv(4096).decode("utf-8")
                response = handle_command(command)
                conn.sendall(response.encode("utf-8"))

In [ ]:
start_server()

In [ ]:
import socket

HOST = "127.0.0.1"
PORT = 5001


def send_command(command):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as client:
        client.connect((HOST, PORT))
        client.sendall(command.encode("utf-8"))

        response = client.recv(1024 * 1024).decode("utf-8")

    return response

In [ ]:
response = send_command("LIST")

print("Files on server:")
print(response)

In [ ]:
response = send_command("GET song1.txt")

print("Downloaded file content:")
print(response)

In [ ]:
response = send_command("GET missing.txt")

print(response)

In [ ]:
response = send_command("DELETE song2.txt")

print(response)

In [ ]:
response = send_command("DELETE unknown.txt")

print(response)

In [ ]:
## Task 5: AI-Generated Code

I asked ChatGPT to generate a DELETE command for the filesystem server.

```python
if action == "DELETE":
    filename = parts[1]
    path = os.path.join(BASE_DIR, filename)

    if os.path.exists(path):
        os.remove(path)
        return f"{filename} deleted"
    else:
        return "File not found"